# N-Gram Analysis Annotated
This notebook aims at investigating the annotations of n-grams.

In [85]:
import os
import pandas as pd
from tqdm import tqdm
import json

df_ngrams_annotated = pd.read_csv(
    os.path.join("data", "tags", "top1000ngrams_annotated.csv"))

df_ngrams_annotated[["category", "ngram"]].groupby("category").agg(list)


,ngram
category,
-,"[youtube, https, com, auto, generated, by_yout..."
artist type - band,"[band, band]"
artist type - dj,[dj]
artist type - orchestra,"[orchestra, his_orchestra, and_his_orchestra]"
artist type - quartet,[quartet]
artist type - singer,[singer]
artist type - trio,[trio]
auto generated,"[auto_generated, generated_by, generated_by_yo..."
composition - focus on lyrics,"[lyricist, lyrics, letra]"


### Cleaning up tags
We just keep annotations with `sonority`, `situational context` or `composition`. 


In [86]:
df_ngrams_annotated.category.unique().tolist()

['-',
 'general ',
 'auto generated',
 'release info',
 'composition - focus on lyrics',
 'instrument vocals - guitar',
 'sonority - live',
 'writer artist mentioned',
 'situational context - cover',
 'genre',
 'release type',
 'music company label',
 'instrument vocals - bass',
 'artist type - band',
 'instrument vocals - piano',
 'sonority - instrumental',
 'artist type - orchestra',
 'instrument vocals - vocals',
 'situational context - studio',
 'instrument vocals - drums',
 'sonority - acoustic',
 'situational context - reaction',
 'segment - solo',
 'visual content - dance',
 'instrument vocals - saxophone',
 'situational context - official',
 'artist type - trio',
 'instrument vocals - trumpet',
 'artist type - singer',
 'situational context - tutorial',
 'instrument vocals - keyboard',
 'instrument vocals - percussion',
 'instrument vocals - violin',
 'sonority - hq',
 'instrument vocals - voice',
 'sonority - mono',
 'instrument vocals - trombone',
 'segment - chorus',
 'instr

In [87]:
keywords = ("sonority", "situational context", "composition", "auto generated", 
            "segment", "instrument vocals", "artist type")

def keep_condition(s):
    if not isinstance(s, str):
        return False
    s_norm = s.strip().lower()
    return any(k in s_norm for k in keywords)

# show matching category names (cleaned)
matching_categories = sorted({cat.strip() for cat in df_ngrams_annotated["category"].unique() if keep_condition(cat)})

# filter dataframe to keep only rows whose category matches the condition
df_tags = df_ngrams_annotated[df_ngrams_annotated["category"].isin(matching_categories)].copy()
df_tags["category"] = df_tags["category"].replace({
    "artist type": "instruments groups",
    "instrument vocals": "instruments groups",
})


df_tags_grouped = df_tags[["category", "ngram"]].groupby("category", as_index=False).agg(list)#.to_dict(orient="index")
df_tags_grouped


,category,ngram
0,artist type - band,"[band, band]"
1,artist type - dj,[dj]
2,artist type - orchestra,"[orchestra, his_orchestra, and_his_orchestra]"
3,artist type - quartet,[quartet]
4,artist type - singer,[singer]
5,artist type - trio,[trio]
6,auto generated,"[auto_generated, generated_by, generated_by_yo..."
7,composition - focus on lyrics,"[lyricist, lyrics, letra]"
8,instrument vocals - acoustic guitar,"[acoustic_guitar, violao]"
9,instrument vocals - bass,"[bass, bass_guitar]"


## Add a few manually found cues

In [ ]:
import yaml

tag_map = dict(zip(df_tags_grouped["category"], df_tags_grouped["ngram"]))

# add manually
tag_map["artist type - band"] = ["band"]
tag_map["artist type - big band"] = ["big band"]
tag_map["segment - verse"] = ["verse"]
tag_map["segment - intro"] = ["intro"]
tag_map["segment - outro"] = ["outro"]
tag_map['situational context - cover'].append("interpretation")
tag_map['situational context - reaction'].append("first_time")
tag_map['situational context - reaction'].append("analysis")
tag_map['situational context - reaction'].append("reviews")
tag_map['situational context - reaction'].append("review")
tag_map["sonority - electric"] = ["electric"]
tag_map['sonority - hq'].append("hq")
tag_map['sonority - hq'].append("high_quality")
tag_map['sonority - hq'].append("high_definition")
tag_map['sonority - instrumental'].append("no_vocals")


out_dir = os.path.join("data", "tags")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "tags.yaml")

with open(out_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(tag_map, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f"Saved tag_map to {out_path}")


Saved tag_map to data/tags/tags.yaml


### Add conjugation and plural variants


### Translate to most frequent European languages
We use *Qwen3* to translate the cues into other languages.


##### Prompt
*You are an expert translator for the domain of online music videos. You are given a mapping of concepts in the domain each mapping to different cues (most of them in English language). Please translate these cues each into the major European languages: Spanish, French, German, Italien, and Portuguese. Avoid translations being too general and make sure that these conform with the domain of online music videos.* 

### Manual cleanups
#### Genres
We realized, that all genres are mentioned amoung the `release_stlye` from *Discogs* which are already included in the *DVI* dataset. Hence, we do not consider our file `genres.yaml` anymore. Furhtermore, we perform some manual and LLM-involving cleanups of `instruments_groups.yaml`.